In [ ]:
!pip install -q -U transformers accelerate

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import re
import json
import glob
import time
import subprocess
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
DT = torch.float16

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=DT,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
    attn_implementation="sdpa",
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

lm = model.model.language_model
lm_head = model.lm_head
layers = lm.layers
n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = hidden_dim // n_head
IMAGE_TOKEN_ID = getattr(model.config, "image_token_id", None) or model.config.image_token_index
assert len(layers) == n_layer
assert model.config.text_config._attn_implementation == "sdpa"
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim, image token id", IMAGE_TOKEN_ID)

In [ ]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"

N_UNITS = 100
DRIVEN_TE = -1.0
SLUG = f"{USERNAME}/patch-llava-intervene"
N_ADD = 8
N_EVAL = 120
EVAL_SPLITS = ["val", "test"]
EVAL_ALPHAS = {"abl": 0.0, "half": 0.5, "up2": 2.0, "up3": 3.0, "up15": 1.5}
MAX_HOURS = 9.0
PILOT_UNITS = 2
BATCH_SAVE_SIZE = 20
UPLOAD_EVERY = 2
CHECK_TOL = 0.25
YES_ID, NO_ID = 3869, 1939
TAIL_N = 4
GROUPS = ["visual", "text", "tail"]
OUT_DIR = "/kaggle/working/patch_llava_intervene"
os.makedirs(OUT_DIR, exist_ok=True)

ROUTERS = [("mlp", 10, -1, "visual"), ("head", 11, 26, "text"), ("head", 12, 13, "text")]
FIXED_CORE = ROUTERS + [("head", 16, 15, "all"), ("head", 16, 0, "all")]
FALLBACK_SURVIVORS = [("mlp", 16, -1, "all"), ("mlp", 17, -1, "all"), ("mlp", 18, -1, "all"), ("mlp", 28, -1, "all"), ("mlp", 31, -1, "all")]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, DT)


def resolve_image_path(q):
    if q["question_id"].startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    return None


def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
assert [processor.tokenizer.decode([i]).strip() for i in (YES_ID, NO_ID)] == ["Yes", "No"]

atp = {}
atp_files = sorted(glob.glob(f"{INPUT}/atp-llava-noising/**/*_batch_*.pt", recursive=True), key=lambda p: (os.path.basename(p).split("_batch_")[0], batch_number(p)))
for path in atp_files:
    for e in torch.load(path):
        if e["ok"]:
            atp[e["question_id"]] = {k: e[k] for k in (
                "question_id", "image_id", "hallucination_type", "stratum", "group", "source", "split", "sign", "seed",
                "delta_clean", "delta_corrupt", "TE", "noise_used", "head", "mlp",
            )}
print(f"atp-llava-noising: {len(atp_files)} batch files, {len(atp)} ok entries")
assert len(atp) >= 0.98 * 8403, "attach atp-llava-noising (all batches)"

meta = pd.DataFrame([{k: v for k, v in a.items() if k not in ("head", "mlp")} for a in atp.values()])
meta["gap_clean"] = meta.sign * meta.delta_clean
fy_all = meta[(meta.stratum == "false_yes") & (meta.hallucination_type == "attribute") & (meta.TE < DRIVEN_TE)]
ty_all = meta[(meta.stratum == "true_yes") & (meta.hallucination_type == "attribute")]
rng = np.random.default_rng(0)
units = []
for (s, sp), g in fy_all.groupby(["source", "split"]):
    pool = ty_all[(ty_all.source == s) & (ty_all.split == sp)].set_index("question_id").gap_clean
    ids = g.question_id.tolist()
    gaps = g.set_index("question_id").gap_clean
    for rank, i in enumerate(rng.permutation(len(ids))):
        qid = ids[i]
        ctrl = None
        if len(pool):
            ctrl = (pool - gaps[qid]).abs().idxmin()
            pool = pool.drop(ctrl)
        units.append({"fy": qid, "ctrl": ctrl, "key": (rank + 0.5) / len(ids), "stratum": f"{s}/{sp}"})
units.sort(key=lambda u: (u["key"], u["stratum"]))
for i, u in enumerate(units):
    u["uid"] = i
print(len(units), "driven attribute false_yes units,", sum(u["ctrl"] is not None for u in units), "with a matched true_yes control")
stage_units = units[:N_UNITS]
uid_of = {}
for u in stage_units:
    uid_of[u["fy"]] = u["uid"]
    if u["ctrl"] is not None:
        uid_of[u["ctrl"]] = u["uid"]
stage_ids = list(uid_of)
print(f"{len(stage_units)} units, {len(stage_ids)} questions")
print(pd.Series([u["stratum"] for u in stage_units]).value_counts().to_string())


def comp_name(kind, layer, head, group):
    base = f"L{layer} H{head}" if kind == "head" else f"L{layer} MLP"
    return f"{base} {group}"


CORE = {}
for c in FIXED_CORE:
    CORE[comp_name(*c)] = c
survivors_path = f"{INPUT}/patch-llava-effects/survivors.json"
if os.path.exists(survivors_path):
    with open(survivors_path) as f:
        survivors = json.load(f)
    for s in survivors.values():
        c = s["cell"]
        CORE[comp_name(c["kind"], c["layer"], c["head"], c["group"])] = (c["kind"], c["layer"], c["head"], c["group"])
    print(len(survivors), "survivors read from patch-llava-effects")
else:
    print("patch-llava-effects not attached, using the fallback survivors")
    for c in FALLBACK_SURVIVORS:
        CORE[comp_name(*c)] = c
CORE_COMPS = list(CORE.values())
print(len(CORE_COMPS), "core circuit components:", list(CORE))

train_fy = [q for q in fy_all[fy_all.split == "train"].question_id if q in atp]
sup_head = np.zeros((len(GROUPS), n_layer, n_head))
sup_mlp = np.zeros((len(GROUPS), n_layer))
for q in train_fy:
    a = atp[q]
    sup_head += (-a["sign"] * a["head"]).numpy()
    sup_mlp += (-a["sign"] * a["mlp"]).numpy()
sup_head /= len(train_fy)
sup_mlp /= len(train_fy)
print(f"signed attribution patching support for Yes on {len(train_fy)} driven train false_yes questions")


def comp_support(c):
    kind, layer, head, group = c
    arr = sup_head[:, layer, head] if kind == "head" else sup_mlp[:, layer]
    return float(arr.sum() if group == "all" else arr[GROUPS.index(group)])


PROMOTERS = [c for c in CORE_COMPS if comp_support(c) > 0]
BRAKES = [c for c in CORE_COMPS if comp_support(c) <= 0]
print("core promoters (positive support):", [(comp_name(*c), round(comp_support(c), 3)) for c in PROMOTERS])
print("core brakes (non-positive support):", [(comp_name(*c), round(comp_support(c), 3)) for c in BRAKES])
core_all = {(k, l, h) for k, l, h, g in CORE_COMPS if g == "all"}
cands = [("head", l, h, "all") for l in range(n_layer) for h in range(n_head)] + [("mlp", l, -1, "all") for l in range(n_layer)]
cands = [c for c in cands if (c[0], c[1], c[2]) not in core_all]
RANKED_POS = [c for c in sorted(cands, key=lambda c: -comp_support(c)) if comp_support(c) > 0]
print("top positive additions:", [(comp_name(*c), round(comp_support(c), 3)) for c in RANKED_POS[:N_ADD]])


def cell_name(entries):
    kind, layer, head = entries[0][0], entries[0][1], entries[0][2]
    groups = {e[3] for e in entries}
    base = f"L{layer} H{head}" if kind == "head" else f"L{layer} MLP"
    return base if len(groups) > 1 or "all" in groups else f"{base}[{next(iter(groups))}]"


core_entries, add_entries = {}, {}
for c in CORE_COMPS:
    core_entries.setdefault(c[:3], []).append(c)
for c in RANKED_POS[:N_ADD]:
    add_entries.setdefault(c[:3], []).append(c)
P_KEYS = [k_ for k_, es in core_entries.items() if any(e in PROMOTERS for e in es)]
B_KEYS = [k_ for k_, es in core_entries.items() if k_ not in P_KEYS]
A_KEYS = [k_ for k_ in add_entries if k_ not in core_entries]
PA_KEYS = P_KEYS + A_KEYS
members = {}
for k_ in P_KEYS:
    members[k_] = core_entries[k_] + add_entries.get(k_, [])
for k_ in A_KEYS:
    members[k_] = add_entries[k_]
for k_ in B_KEYS:
    members[k_] = core_entries[k_]
NAME = {k_: cell_name(members[k_]) for k_ in members}
ROLE = {k_: "promoter" if k_ in P_KEYS else "brake" if k_ in B_KEYS else "addition" for k_ in members}
print("promoter cells:", [NAME[k_] for k_ in P_KEYS])
print("brake cells:", [NAME[k_] for k_ in B_KEYS])
print("addition cells:", [NAME[k_] for k_ in A_KEYS])


def image_ok(qid):
    path = resolve_image_path(question_by_id[qid])
    return path is not None and os.path.exists(path)


def comps_of(keys, skip=None):
    return [e for k_ in keys if k_ != skip for e in members[k_]]


BYNAME = {NAME[k_]: members[k_] for k_ in members}


def cells(*names):
    return [e for n in names for e in BYNAME[n]]


M2 = cells("L11 H26", "L16 MLP", "L17 MLP")
L28 = cells("L28 MLP")
SETS = {
    "L16+L17": cells("L16 MLP", "L17 MLP"),
    "M2": M2,
    "M2+L28": M2 + L28,
    "M2+L28+L12H13": M2 + L28 + cells("L12 H13[text]"),
    "M5": M2 + cells("L30 MLP", "L13 MLP"),
    "M5+L28": M2 + cells("L30 MLP", "L13 MLP") + L28,
    "P": [e for k_ in P_KEYS for e in core_entries[k_]],
    "P+L28": [e for k_ in P_KEYS for e in core_entries[k_]] + L28,
    "PA-L18": comps_of([k_ for k_ in PA_KEYS if NAME[k_] != "L18 MLP"]),
    "PA": comps_of(PA_KEYS),
}
STEP_COMPS = SETS
STEP_NAMES = list(SETS)
print(len(STEP_NAMES), "circuit sets:", {n: len({c[:3] for c in SETS[n]}) for n in STEP_NAMES})

ctrl_rng = np.random.default_rng(0)
ctrl_layers = [l for l in range(8, 26) if l not in (16, 17, 18)]
CTRL = [sorted(int(x) for x in ctrl_rng.choice(ctrl_layers, 3, replace=False)) for _ in range(2)]
INTERV = {
    "base": [],
    "abl L17": [(cells("L17 MLP"), EVAL_ALPHAS["abl"])],
    "abl L16+L17": [(cells("L16 MLP", "L17 MLP"), EVAL_ALPHAS["abl"])],
    "abl M2": [(M2, EVAL_ALPHAS["abl"])],
    "x0.5 M2": [(M2, EVAL_ALPHAS["half"])],
    "L28 x2": [(L28, EVAL_ALPHAS["up2"])],
    "L28 x3": [(L28, EVAL_ALPHAS["up3"])],
    "abl M2 + L28 x2": [(M2, EVAL_ALPHAS["abl"]), (L28, EVAL_ALPHAS["up2"])],
    "x0.5 M2 + L28 x1.5": [(M2, EVAL_ALPHAS["half"]), (L28, EVAL_ALPHAS["up15"])],
    "ctrl abl A": [([("mlp", l, -1, "all") for l in CTRL[0]], EVAL_ALPHAS["abl"])],
    "ctrl abl B": [([("mlp", l, -1, "all") for l in CTRL[1]], EVAL_ALPHAS["abl"])],
}
INTERV_NAMES = list(INTERV)
print("control layers:", CTRL)

ev = meta[(meta.hallucination_type == "attribute") & meta.split.isin(EVAL_SPLITS)]
eval_groups = {
    "fy_driven": ev[(ev.stratum == "false_yes") & (ev.TE < DRIVEN_TE)],
    "fy_prior": ev[(ev.stratum == "false_yes") & (ev.TE >= DRIVEN_TE)],
    "true_yes": ev[ev.stratum == "true_yes"],
    "true_no": ev[ev.stratum == "true_no"],
}
ev_rng = np.random.default_rng(0)
EVAL, EVAL_GROUP = [], {}
for g_, df_ in eval_groups.items():
    ids_ = [q for q in df_.question_id.tolist() if image_ok(q)]
    ids_ = [ids_[i] for i in ev_rng.permutation(len(ids_))[:N_EVAL]]
    for q in ids_:
        EVAL.append(q)
        EVAL_GROUP[q] = g_
    print(f"eval group {g_}: {len(df_)} available, {len(ids_)} used")
print(len(EVAL), "eval questions,", len(INTERV_NAMES), "interventions each")


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == 576 and not bool((vis & tail).any())
    return {
        "qid": qid, "T": T, "input_ids": ids, "pixel_values": inputs["pixel_values"],
        "attention_mask": inputs["attention_mask"], "vis": vis,
        "idx": {
            "all": torch.arange(T), "visual": torch.nonzero(vis.cpu())[:, 0],
            "text": torch.nonzero(text.cpu())[:, 0], "tail": torch.nonzero(tail.cpu())[:, 0],
        },
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }

In [ ]:
STATE = {"noise": None, "seed": 0, "capture": False, "live": None, "vis": None, "freeze": None, "scale": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        if l != 0 or STATE["noise"] is None:
            return None
        h = args[0] if args else kwargs["hidden_states"]
        vis = STATE["vis"]
        h = h.clone()
        x = h[:, vis].float()
        generator = torch.Generator().manual_seed(STATE["seed"])
        noise = torch.randn(x.shape, generator=generator).to(h.device)
        h[:, vis] = (x + noise * x.std() * STATE["noise"]).to(h.dtype)
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        z = args[0]
        sc = STATE["scale"]
        if sc is not None:
            m = sc["z"][l]
            if m is None:
                return None
            B, T = z.shape[0], z.shape[1]
            z4 = z.view(B, T, n_head, head_dim) * m.to(z.device, z.dtype)[None, :, :, None]
            return (z4.view(B, T, n_head * head_dim),) + tuple(args[1:])
        fz = STATE["freeze"]
        if fz is not None:
            m = fz["z"][l]
            if m is None:
                return None
            B, T = z.shape[0], z.shape[1]
            src = fz["src"]["z"][l][0].view(T, n_head, head_dim).to(z.device, z.dtype)
            z4 = torch.where(m.to(z.device)[None, :, :, None], src[None], z.view(B, T, n_head, head_dim))
            return (z4.view(B, T, n_head * head_dim),) + tuple(args[1:])
        if STATE["capture"]:
            STATE["live"]["z"][l] = z
        return None
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        sc = STATE["scale"]
        if sc is not None:
            m = sc["m"][l]
            if m is None:
                return None
            return output * m.to(output.device, output.dtype)[None, :, None]
        fz = STATE["freeze"]
        if fz is not None:
            m = fz["m"][l]
            if m is None:
                return None
            src = fz["src"]["mlp"][l][0].to(output.device, output.dtype)
            return torch.where(m.to(output.device)[None, :, None], src[None], output)
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = output
        return None
    return hook


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
print(len(_handles), "hooks registered")

In [ ]:
@torch.no_grad()
def run_forward(p, noise, seed, freeze=None, capture=False, scale=None):
    STATE.update(noise=noise, seed=seed, capture=capture, freeze=freeze, vis=p["vis"], scale=scale)
    if capture:
        STATE["live"] = {"z": [None] * n_layer, "mlp": [None] * n_layer}
    out = model(input_ids=p["input_ids"], pixel_values=p["pixel_values"], attention_mask=p["attention_mask"], use_cache=False, logits_to_keep=1)
    logits = out.logits[0, -1].float()
    delta = float(p["sign"] * (logits[YES_ID] - logits[NO_ID]))
    result = {"acts": STATE["live"], "delta": delta} if capture else delta
    STATE.update(noise=None, capture=False, freeze=None, live=None, scale=None)
    del out, logits
    return result


def build_mask(p, comps):
    T = p["T"]
    hm = torch.zeros(n_layer, T, n_head, dtype=torch.bool)
    mm = torch.zeros(n_layer, T, dtype=torch.bool)
    for kind, layer, head, group in comps:
        idx = p["idx"][group]
        if kind == "head":
            hm[layer, idx, head] = True
        else:
            mm[layer, idx] = True
    return hm, mm


def make_freeze(hm, mm, src, invert):
    zs, ms = [], []
    for l in range(n_layer):
        a, b = hm[l], mm[l]
        if invert:
            a, b = ~a, ~b
        zs.append(a if bool(a.any()) else None)
        ms.append(b if bool(b.any()) else None)
    return {"z": zs, "m": ms, "src": src["acts"]}


def build_scale(p, spec, force=False):
    T = p["T"]
    mz = torch.ones(n_layer, T, n_head)
    mm_ = torch.ones(n_layer, T)
    for comps, alpha in spec:
        hm, mm = build_mask(p, comps)
        mz[hm] = alpha
        mm_[mm] = alpha
    zs = [mz[l] if (force or bool((mz[l] != 1).any())) else None for l in range(n_layer)]
    ms = [mm_[l] if (force or bool((mm_[l] != 1).any())) else None for l in range(n_layer)]
    return {"z": zs, "m": ms}


def intervene(p, spec, force=False):
    scale = build_scale(p, spec, force) if (spec or force) else None
    return p["sign"] * run_forward(p, None, 0, scale=scale)


def two_configs(p, clean, corrupt, comps, seed, noise):
    hm, mm = build_mask(p, comps)
    knock = run_forward(p, None, seed, make_freeze(hm, mm, corrupt, False))
    restore = run_forward(p, noise, seed, make_freeze(hm, mm, clean, False))
    return [knock, restore]


CONFIG_NAMES = ["knockout", "restored"]


def circuit_curve(p, clean, corrupt, seed, noise):
    return np.array([two_configs(p, clean, corrupt, STEP_COMPS[n], seed, noise) for n in STEP_NAMES], dtype=np.float32)


ALL_COMPS = [("head", l, h, "all") for l in range(n_layer) for h in range(n_head)] + [("mlp", l, -1, "all") for l in range(n_layer)]

In [ ]:
pilot_ids = [q for u in stage_units for q in (u["fy"], u["ctrl"]) if q is not None and image_ok(q)][:2 * PILOT_UNITS]
secs_a, base_err, id_err, plain_err, force_err = [], [], [], [], []
for qid in pilot_ids:
    a = atp[qid]
    p = prep(qid)
    t0 = time.time()
    clean = run_forward(p, None, a["seed"], capture=True)
    corrupt = run_forward(p, a["noise_used"], a["seed"], capture=True)
    base_err.append(max(abs(clean["delta"] - a["delta_clean"]), abs(corrupt["delta"] - a["delta_corrupt"])))
    plain_err.append(max(abs(run_forward(p, None, a["seed"]) - clean["delta"]), abs(run_forward(p, a["noise_used"], a["seed"]) - corrupt["delta"])))
    all_knock, all_restore = two_configs(p, clean, corrupt, ALL_COMPS, a["seed"], a["noise_used"])
    none_knock, none_restore = two_configs(p, clean, corrupt, [], a["seed"], a["noise_used"])
    id_err.append(max(abs(all_knock - corrupt["delta"]), abs(all_restore - clean["delta"]), abs(none_knock - clean["delta"]), abs(none_restore - corrupt["delta"])))
    force_err.append(abs(p["sign"] * intervene(p, [], force=True) - clean["delta"]))
    curve = circuit_curve(p, clean, corrupt, a["seed"], a["noise_used"])
    secs_a.append(time.time() - t0)
    print(f"{qid} {a['stratum']:9s} T={p['T']}: {secs_a[-1]:.1f} s | clean {clean['delta']:+.2f} (atp run {a['delta_clean']:+.2f})")
    print(pd.DataFrame(curve, index=STEP_NAMES, columns=CONFIG_NAMES).round(2).to_string())
    del p, clean, corrupt
    torch.cuda.empty_cache()
print(f"clean and corrupt deltas vs the attribution patching run: max abs diff {max(base_err):.3f}")
print(f"capture runs vs plain runs: max abs diff {max(plain_err):.3f}")
print(f"identity checks (freezing every component, or none): max abs error {max(id_err):.3f}")
print(f"scale hooks with all multipliers at 1 vs the plain run: max abs error {max(force_err):.3f}")
assert max(base_err) < CHECK_TOL and max(plain_err) < CHECK_TOL and max(id_err) < CHECK_TOL and max(force_err) < CHECK_TOL, "engine checks failed"

secs_b = []
for qid in EVAL[:2] + EVAL[-2:]:
    p = prep(qid)
    t0 = time.time()
    margins = [intervene(p, INTERV[n]) for n in INTERV_NAMES]
    secs_b.append(time.time() - t0)
    print(f"{qid} {EVAL_GROUP[qid]:9s} T={p['T']}: {secs_b[-1]:.1f} s | yes minus no", {n: round(m, 2) for n, m in zip(INTERV_NAMES, margins)})
    del p
    torch.cuda.empty_cache()
n_a = len(stage_ids)
print(f"stage A: {np.mean(secs_a):.1f} s per question, projected {np.mean(secs_a) * n_a / 3600:.1f} h for {n_a} questions")
print(f"stage B: {np.mean(secs_b):.1f} s per question, projected {np.mean(secs_b) * len(EVAL) / 3600:.1f} h for {len(EVAL)} questions")

In [ ]:
def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    cleaned = "\n".join(l for l in out.splitlines() if "upload info" not in l.lower()).lower()
    bad = ("client error", "forbidden", "unauthorized", "traceback", "error:", "failed")
    return r.returncode == 0 and not any(x in cleaned for x in bad), out


ok, out = _run(["kaggle", "datasets", "download", "-d", SLUG, "-p", OUT_DIR, "--unzip"])
if ok:
    print("restored", len(glob.glob(os.path.join(OUT_DIR, "C_batch_*.pt"))), "batch files from", SLUG)
else:
    if not any(s in out.lower() for s in ("404", "403", "not found", "does not exist", "forbidden")):
        raise RuntimeError("download failed for an unrelated reason, refusing to start fresh:\n" + out)
    print("no existing dataset, starting fresh")
dataset_exists = ok

with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": SLUG.split("/")[1],
        "id": SLUG,
        "subtitle": "LLaVA minimal circuit sets and ablation interventions",
        "description": "LLaVA-1.5-7B attribute questions. A_batch files: for driven false_yes questions (TE below -1) and matched true_yes controls from atp-llava-noising, curve [set, 2] with the sign-corrected Yes minus No gap for knockout (clean input, set takes corrupt activations) and restored (corrupt input, set takes clean activations). "
                       "B_batch files: for held-out val and test attribute questions in four groups (driven false_yes, prior false_yes, true_yes, true_no), margins [intervention] holding Yes minus No logits at the last position under scaling of the output of chosen MLPs and heads (0 ablates, above 1 amplifies). Names are in circuit_meta.json.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def upload(msg):
    global dataset_exists
    version_cmd = ["kaggle", "datasets", "version", "-p", OUT_DIR, "-m", msg]
    create_cmd = ["kaggle", "datasets", "create", "-p", OUT_DIR]
    ok_, out_ = _run(version_cmd if dataset_exists else create_cmd)
    if not ok_ and not dataset_exists:
        time.sleep(30)
        ok_, out2 = _run(version_cmd)
        out_ += "\n--- version retry ---\n" + out2
    if ok_:
        dataset_exists = True
        print(f"    [upload ok] {msg}")
    else:
        cleaned = [l for l in out_.splitlines() if l.strip() and "upload info" not in l.lower() and "B/s]" not in l and "it/s]" not in l]
        print(f"    [UPLOAD FAILED, continuing] account {USERNAME}, dataset {SLUG}\n" + "\n".join(cleaned[-8:]))
    return ok_


with open(os.path.join(OUT_DIR, "circuit_meta.json"), "w") as f:
    json.dump({
        "set_names": STEP_NAMES, "sets": {n: [list(c) for c in SETS[n]] for n in STEP_NAMES}, "config_names": CONFIG_NAMES,
        "interventions": {n: [[[list(c) for c in comps], alpha] for comps, alpha in INTERV[n]] for n in INTERV_NAMES}, "intervention_names": INTERV_NAMES,
        "control_layers": CTRL, "eval_group": EVAL_GROUP, "n_units": N_UNITS, "n_eval": N_EVAL, "tail_n": TAIL_N, "yes_id": YES_ID, "no_id": NO_ID,
    }, f)


def collect(tag, ids, make_entry):
    paths = sorted(glob.glob(os.path.join(OUT_DIR, f"{tag}_batch_*.pt")), key=batch_number)
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [q for q in ids if q not in done_ids]
    print(f"{tag}: {len(ids)} questions, {len(done_ids)} already done, {len(todo)} to run")
    batch_entries, batch_index, failed, missing = [], len(paths), 0, 0
    t_start = time.time()
    stopped = False
    for i, qid in enumerate(todo):
        if not image_ok(qid):
            missing += 1
            continue
        try:
            entry = make_entry(qid)
        except torch.cuda.OutOfMemoryError:
            STATE.update(capture=False, live=None, noise=None, freeze=None, scale=None)
            torch.cuda.empty_cache()
            entry = {"question_id": qid, "ok": False, "reason": "oom"}
            failed += 1
        batch_entries.append(entry)
        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1
            if batch_index % UPLOAD_EVERY == 0:
                upload(f"{tag}: {batch_index} batches")
        if i % 10 == 0 and i > 0:
            torch.cuda.empty_cache()
            avg = (time.time() - t_start) / (i + 1)
            print(f"[{tag} {i + 1}/{len(todo)}] {qid} ({avg:.1f} s/question avg, ETA {avg * (len(todo) - i - 1) / 3600:.1f} h, failed {failed}, missing images {missing})")
        if time.time() - t_start > MAX_HOURS * 3600:
            stopped = True
            print(f"time budget of {MAX_HOURS} h reached after {i + 1} questions; re-run this notebook to resume")
            break
    if batch_entries:
        torch.save(batch_entries, os.path.join(OUT_DIR, f"{tag}_batch_{batch_index}.pt"))
    upload(f"{tag}: " + ("paused" if stopped else "done"))
    print(f"{tag} done, failed {failed}, missing images {missing}, {(time.time() - t_start) / 3600:.2f} h, complete {not stopped}")


def make_entry_a(qid):
    a = atp[qid]
    q = question_by_id[qid]
    p = prep(qid)
    clean = run_forward(p, None, a["seed"], capture=True)
    corrupt = run_forward(p, a["noise_used"], a["seed"], capture=True)
    curve = circuit_curve(p, clean, corrupt, a["seed"], a["noise_used"])
    entry = {
        "question_id": qid, "uid": uid_of[qid], "stratum": a["stratum"], "source": a["source"], "split": a["split"], "sign": p["sign"],
        "ok": True, "T": p["T"], "delta_clean": clean["delta"], "delta_corrupt": corrupt["delta"], "curve": torch.from_numpy(curve),
    }
    del p, clean, corrupt
    return entry


def make_entry_b(qid):
    a = atp[qid]
    p = prep(qid)
    margins = [intervene(p, INTERV[n]) for n in INTERV_NAMES]
    entry = {
        "question_id": qid, "group": EVAL_GROUP[qid], "stratum": a["stratum"], "split": a["split"], "sign": p["sign"], "TE": a["TE"],
        "ok": True, "T": p["T"], "margins": torch.tensor(margins, dtype=torch.float32),
    }
    del p
    return entry


collect("A", stage_ids, make_entry_a)
collect("B", EVAL, make_entry_b)

In [ ]:
def load_tag(tag):
    out = []
    for path in sorted(glob.glob(os.path.join(OUT_DIR, f"{tag}_batch_*.pt")), key=batch_number):
        out.extend(torch.load(path))
    return out


rng = np.random.default_rng(1)
N_BOOT = 1000

entries_a = load_tag("A")
good = [e for e in entries_a if e["ok"] and bool(np.isfinite(e["curve"].numpy()).all())]
assert len({e["question_id"] for e in good}) == len(good), "duplicated entries"
print(f"stage A: {len(entries_a)} entries, {len(good)} ok and finite, {len(stage_ids) - len(good)} stage questions not done")
CV = np.stack([e["curve"].numpy() for e in good]).astype(np.float64)
dc = np.array([e["delta_clean"] for e in good])
dk = np.array([e["delta_corrupt"] for e in good])
sign = np.array([e["sign"] for e in good])
role = np.array([e["stratum"] for e in good])
uid = np.array([e["uid"] for e in good])
fy = np.flatnonzero(role == "false_yes")
ct = np.flatnonzero(role == "true_yes")
S = len(STEP_NAMES)
si = {n: i for i, n in enumerate(STEP_NAMES)}
n_dist = [len({c[:3] for c in STEP_COMPS[n]}) for n in STEP_NAMES]


def share_sums(ix):
    den = (dk[ix] - dc[ix]).sum()
    knock = (CV[ix, :, 0] - dc[ix, None]).sum(0) / den
    restored = (CV[ix, :, 1] - dk[ix, None]).sum(0) / -den
    return np.stack([knock, restored], 1)


point = share_sums(fy)
draws = rng.integers(0, len(fy), size=(N_BOOT, len(fy)))
boots = np.stack([share_sums(fy[d]) for d in draws])
lo, hi = np.percentile(boots, 2.5, axis=0), np.percentile(boots, 97.5, axis=0)

pairs = sorted(set(uid[fy]) & set(uid[ct]))
fy_ix = {u: i for u, i in zip(uid[fy], fy)}
ct_ix = {u: i for u, i in zip(uid[ct], ct)}
a_ix = np.array([fy_ix[u] for u in pairs])
b_ix = np.array([ct_ix[u] for u in pairs])
pdraws = rng.integers(0, len(pairs), size=(N_BOOT, len(pairs)))
sup = [-sign[:, None] * (CV[:, :, 0] - dc[:, None]), sign[:, None] * (CV[:, :, 1] - dk[:, None])]
spec = []
for arr in sup:
    diff = arr[a_ix] - arr[b_ix]
    means = diff[pdraws].mean(1)
    spec.append((diff.mean(0), np.percentile(means, 2.5, axis=0), np.percentile(means, 97.5, axis=0)))
fix = (CV[a_ix, :, 0] > 0).astype(float)
hurt = (CV[b_ix, :, 0] < 0).astype(float)
fix_m, fix_ci = fix.mean(0), np.percentile(fix[pdraws].mean(1), [2.5, 97.5], axis=0)
hurt_m, hurt_ci = hurt.mean(0), np.percentile(hurt[pdraws].mean(1), [2.5, 97.5], axis=0)

tab = pd.DataFrame({"cells": n_dist}, index=STEP_NAMES)
tab["knockout"] = [f"{point[i, 0]:+.2f} [{lo[i, 0]:+.2f}, {hi[i, 0]:+.2f}]" for i in range(S)]
tab["restored"] = [f"{point[i, 1]:+.2f} [{lo[i, 1]:+.2f}, {hi[i, 1]:+.2f}]" for i in range(S)]
tab["spec knockout"] = [f"{spec[0][0][i]:+.3f} [{spec[0][1][i]:+.3f}, {spec[0][2][i]:+.3f}]" for i in range(S)]
tab["spec restored"] = [f"{spec[1][0][i]:+.3f} [{spec[1][1][i]:+.3f}, {spec[1][2][i]:+.3f}]" for i in range(S)]
tab["false_yes fixed"] = [f"{fix_m[i]:.2f} [{fix_ci[0][i]:.2f}, {fix_ci[1][i]:.2f}]" for i in range(S)]
tab["true_yes hurt"] = [f"{hurt_m[i]:.2f} [{hurt_ci[0][i]:.2f}, {hurt_ci[1][i]:.2f}]" for i in range(S)]
print(f"--- stage A: circuit sets over {len(fy)} driven false_yes questions and {len(pairs)} matched units ---")
print(tab.to_string())
print("fixed = share of false_yes questions whose answer flips to No when the set takes its corrupt-run activations; hurt = share of matched true_yes questions that flip to No")

entries_b = [e for e in load_tag("B") if e["ok"]]
assert len({e["question_id"] for e in entries_b}) == len(entries_b), "duplicated entries"
M = np.stack([e["margins"].numpy() for e in entries_b]).astype(np.float64)
grp = np.array([e["group"] for e in entries_b])
K = len(INTERV_NAMES)
groups = [g for g in ["fy_driven", "fy_prior", "true_yes", "true_no"] if (grp == g).any()]
pred_yes = M > 0
correct = np.zeros_like(pred_yes)
for g in groups:
    ix = grp == g
    correct[ix] = pred_yes[ix] if g == "true_yes" else ~pred_yes[ix]
print(f"--- stage B: {len(entries_b)} held-out questions, groups {dict(zip(*np.unique(grp, return_counts=True)))} ---")
print("baseline accuracy per group (false_yes is 0 and the true groups are 1 by construction):", {g: round(float(correct[grp == g, 0].mean()), 2) for g in groups})


def boot_mean(x, nb=N_BOOT):
    d = rng.integers(0, len(x), size=(nb, len(x)))
    return np.percentile(x[d].mean(1), [2.5, 97.5], axis=0)


acc = pd.DataFrame(index=INTERV_NAMES)
for g in groups:
    c = correct[grp == g].astype(float)
    ci = boot_mean(c)
    acc[g] = [f"{c[:, k].mean():.2f} [{ci[0][k]:.2f}, {ci[1][k]:.2f}]" for k in range(K)]
macro_groups = [g for g in ("fy_driven", "true_yes", "true_no") if g in groups]
macro_draws = np.stack([rng.integers(0, (grp == g).sum(), size=(N_BOOT, (grp == g).sum())) for g in macro_groups], 0) if False else None
macro_pt = np.mean([correct[grp == g].astype(float).mean(0) for g in macro_groups], axis=0)
macro_boot = np.mean([np.stack([correct[grp == g].astype(float)[rng.integers(0, (grp == g).sum(), size=(grp == g).sum())].mean(0) for _ in range(N_BOOT)]) for g in macro_groups], axis=0)
mlo, mhi = np.percentile(macro_boot, 2.5, axis=0), np.percentile(macro_boot, 97.5, axis=0)
acc["macro"] = [f"{macro_pt[k]:.2f} [{mlo[k]:.2f}, {mhi[k]:.2f}]" for k in range(K)]
print("--- accuracy per group under each intervention (false_yes: share moved to No; true_yes: share kept Yes; true_no: share kept No), macro is the mean over fy_driven, true_yes and true_no ---")
print(acc.to_string())

shift = pd.DataFrame(index=INTERV_NAMES)
for g in groups:
    shift[g] = [f"{(M[grp == g, k] - M[grp == g, 0]).mean():+.2f}" for k in range(K)]
print("--- mean change in the Yes minus No logit gap against the baseline run, logits ---")
print(shift.to_string())

BLUE, ORANGE, GREY, GREEN = "#2a78d6", "#eb6834", "#999999", "#3a9d5d"
fig, ax = plt.subplots(figsize=(13, 5))
xs = np.arange(K)
w = 0.8 / max(len(macro_groups), 1)
for j, g in enumerate(macro_groups):
    vals = [correct[grp == g, k].astype(float).mean() for k in range(K)]
    ax.bar(xs + (j - (len(macro_groups) - 1) / 2) * w, vals, width=w, color=[BLUE, GREEN, ORANGE][j], label=g)
ax.set_xticks(xs)
ax.set_xticklabels(INTERV_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("share answered correctly")
ax.legend(frameon=False)
ax.grid(color="#e5e5e2", lw=0.6, axis="y")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "intervention.png"), dpi=150)
plt.show()

with open(os.path.join(OUT_DIR, "intervene_summary.json"), "w") as f:
    json.dump({
        "set_names": STEP_NAMES, "shares": point.tolist(), "ci_lo": lo.tolist(), "ci_hi": hi.tolist(),
        "spec_knockout": spec[0][0].tolist(), "spec_restored": spec[1][0].tolist(), "fixed": fix_m.tolist(), "hurt": hurt_m.tolist(),
        "interventions": INTERV_NAMES, "groups": groups, "accuracy": {g: correct[grp == g].astype(float).mean(0).tolist() for g in groups},
        "macro": macro_pt.tolist(), "n_b": int(len(entries_b)), "n_a": int(len(good)),
    }, f)
if not upload("summary"):
    print(f"final upload failed; the batch files and summary are still in {OUT_DIR}, which is saved as this notebook's output")